# 2.1 — Oversampling with Augmentation for Class Imbalance

Apply **repeat-factor-style oversampling** with per-copy augmentation to force the model to see more minority-class (`no-helmet`, `no-vest`) examples.

**Strategy:**
- Images containing `no-helmet` (class 1): duplicated 10×, each copy augmented differently
- Images containing `no-vest` (class 2): duplicated 3×, each copy augmented differently
- All other images kept as-is
- Combined with YOLOv8's built-in `copy_paste` augmentation and focal loss at training time

## Setup

In [ ]:
# @title Install dependencies
!pip install -q ultralytics roboflow loguru typer python-dotenv pyyaml matplotlib opencv-python-headless albumentations

In [ ]:
# @title Mount Drive or clone repo
import os
from pathlib import Path
import sys

# Option A: Mount Drive
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_DIR = Path("/content/drive/MyDrive/AlGear")

# Option B: Clone from GitHub
!git clone https://github.com/Hndra04/AlGear
PROJECT_DIR = Path("/content/AlGear")

os.chdir(str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR))
print(f"Project root: {PROJECT_DIR}")

In [ ]:
# @title Set Roboflow API key
import os
os.environ["ROBOFLOW_API_KEY"] = ""

from algear.config import ROBOFLOW_DIR, ROBOFLOW_API_KEY
print(f"API key set: {bool(ROBOFLOW_API_KEY)}")
print(f"Dataset at: {ROBOFLOW_DIR}")

In [ ]:
# @title Download dataset (if not already present)
if not ROBOFLOW_DIR.exists():
    from algear.dataset import download_roboflow
    download_roboflow(output_dir=ROBOFLOW_DIR)
else:
    print(f"Dataset already exists at {ROBOFLOW_DIR}")

## 1. Class Distribution Before Oversampling

In [ ]:
# @title Visualise original class distribution
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import yaml

from algear.config import ROBOFLOW_DIR

with open(ROBOFLOW_DIR / "data.yaml") as f:
    data_cfg = yaml.safe_load(f)
class_names = data_cfg["names"]

def count_instances(lbl_dir) -> Counter:
    c = Counter()
    for lbl in sorted(lbl_dir.glob("*.txt")):
        with open(lbl) as f:
            for line in f:
                c[int(line.strip().split()[0])] += 1
    return c

train_counts = count_instances(ROBOFLOW_DIR / "train" / "labels")
total = sum(train_counts.values())

fig, ax = plt.subplots(figsize=(10, 4))
names = [class_names[i] for i in range(len(class_names))]
counts = [train_counts.get(i, 0) for i in range(len(class_names))]
colors = plt.cm.Reds(np.linspace(0.3, 1.0, len(names)))

bars = ax.bar(names, counts, color=colors)
ax.set_title(f"Original Training Set — {total} instances")
ax.set_ylabel("Instance count")
ax.tick_params(axis="x", rotation=30, labelsize=9)
for bar, c in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
            str(c), ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

print(f"Imbalance ratio (helmet:no-helmet):  {counts[0] / max(counts[1], 1):.1f}x")
print(f"Imbalance ratio (vest:no-vest):      {counts[4] / max(counts[2], 1):.1f}x")

## 2. Generate Oversampled Dataset

We scan the training labels, find every image that contains `no-helmet` (class 1) or `no-vest` (class 2), and create augmented copies.

Each copy gets a **different random augmentation** drawn from:
- Brightness / contrast shift
- Hue / saturation / value jitter
- Gaussian blur
- Gaussian noise
- CLAHE (local contrast enhancement)
- Gamma correction
- Grayscale conversion

These are all **box-preserving** transforms — label coordinates stay valid.

In [ ]:
# @title Run oversampling
from algear.dataset import oversample

oversample(
    labels_dir=ROBOFLOW_DIR / "train" / "labels",
    img_dir=ROBOFLOW_DIR / "train" / "images",
    output_dir=PROJECT_DIR / "data" / "processed" / "construction-site-safety-oversampled",
    multiplier_no_helmet=10,
    multiplier_no_vest=3,
)

## 3. Class Distribution After Oversampling

In [ ]:
# @title Visualise oversampled class distribution
oversampled_dir = PROJECT_DIR / "data" / "processed" / "construction-site-safety-oversampled"

os_counts = count_instances(oversampled_dir / "train" / "labels")
os_total = sum(os_counts.values())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

os_counts_list = [os_counts.get(i, 0) for i in range(len(class_names))]

ax1.bar(names, counts, color=colors, alpha=0.7)
ax1.set_title(f"Original — {total} instances")
ax1.set_ylabel("Instance count")
ax1.tick_params(axis="x", rotation=30, labelsize=9)

ax2.bar(names, os_counts_list, color=colors, alpha=0.7)
ax2.set_title(f"Oversampled — {os_total} instances")
ax2.set_ylabel("Instance count")
ax2.tick_params(axis="x", rotation=30, labelsize=9)

for ax, cts in [(ax1, counts), (ax2, os_counts_list)]:
    for bar, c in zip(ax.containers[0], cts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
                str(c), ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()

print(f"Imbalance ratio (helmet:no-helmet):  {os_counts_list[0] / max(os_counts_list[1], 1):.1f}x")
print(f"Imbalance ratio (vest:no-vest):      {os_counts_list[4] / max(os_counts_list[2], 1):.1f}x")

## 4. Train with Oversampled Data + Augmentation

Two layers of imbalance mitigation are stacked:
1. **Pre-augmented oversampling** (just generated)
2. **`copy_paste=0.3`** — YOLOv8 pastes object instances across images during training

YOLOv8's built-in mosaic, flip, HSV jitter, scale, and translate augmentations are also active (default), stacking on top of the pre-augmented copies.

In [ ]:
# @title Train YOLOv8s with imbalance handling
import torch

device = 0 if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

from algear.modeling.train import train_yolov8

results = train_yolov8(
    model_name="yolov8s.pt",
    epochs=50,
    imgsz=640,
    batch=16,
    device=device,
    output_dir=PROJECT_DIR / "models",
    oversample_data=True,
    copy_paste=0.3,
)

## 5. Evaluate on Test Set

In [ ]:
# @title Evaluate best model on test split
from algear.modeling.train import evaluate

# Find the latest trained model
import glob
model_dirs = sorted(PROJECT_DIR.glob("models/*/weights/best.pt"))
latest_model = model_dirs[-1] if model_dirs else None
print(f"Evaluating: {latest_model}")

metrics = evaluate(
    model_path=latest_model,
    data_yaml=ROBOFLOW_DIR / "data.yaml",
    split="test",
    device=device,
)

## 6. Results Summary

Compare against the baseline (no imbalance handling):

| Class | Baseline mAP@50 | Oversampled mAP@50 | Δ |
|---|---|---|---|
| helmet | 0.929 | | |
| no-helmet | 0.592 | | |
| no-vest | 0.630 | | |
| person | 0.890 | | |
| vest | 0.850 | | |
| **Overall** | **0.778** | | |

*(Fill in results after training completes)*